# 1. Advanced MySQL Topics

This notebook covers advanced MySQL concepts:
- Indexes and Query Optimization
- Views
- Stored Procedures
- Transactions and ACID
- Triggers
- Common Table Expressions (CTEs)

# 2. Indexes

**Indexes** speed up data retrieval by creating a quick lookup structure (like a book's index).

## 2.1 Types of Indexes

| Type | Description | Use Case |
|------|-------------|----------|
| `PRIMARY KEY` | Unique, auto-indexed | Row identifier |
| `UNIQUE INDEX` | No duplicates allowed | Email, username |
| `INDEX` (B-Tree) | Standard index | Frequently searched columns |
| `FULLTEXT` | Text search | Searching in text content |
| `COMPOSITE` | Multiple columns | Multi-column searches |

In [ ]:
import mysql.connector

DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'your_password',
    'database': 'test_db'
}

def execute_sql(query, fetch=False):
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)
    if fetch:
        result = cursor.fetchall()
        cursor.close()
        conn.close()
        return result
    conn.commit()
    cursor.close()
    conn.close()

print("Helper function ready!")

In [ ]:
# Creating Indexes

# SQL Examples for creating indexes:
index_examples = """
-- Create index on single column
CREATE INDEX idx_email ON employees(email);

-- Create unique index
CREATE UNIQUE INDEX idx_unique_email ON employees(email);

-- Create composite index (multiple columns)
CREATE INDEX idx_dept_salary ON employees(department, salary);

-- Create index with specific length for TEXT columns
CREATE INDEX idx_name ON employees(name(50));

-- View existing indexes
SHOW INDEX FROM employees;

-- Drop an index
DROP INDEX idx_email ON employees;
"""

print("Index SQL Examples:")
print(index_examples)

In [ ]:
# When to use indexes (and when not to)

print("""WHEN TO USE INDEXES:
========================

DO use indexes on:
- Columns frequently used in WHERE clauses
- Columns used in JOIN conditions
- Columns used in ORDER BY
- Foreign key columns
- Columns with high cardinality (many unique values)

DON'T use indexes on:
- Small tables (full scan may be faster)
- Columns rarely used in queries
- Columns with low cardinality (few unique values like gender, status)
- Tables with frequent INSERT/UPDATE/DELETE (indexes slow writes)

TRADE-OFF:
- Indexes speed up READ operations
- Indexes slow down WRITE operations (must update index)
- Indexes consume additional storage space
""")

## 2.2 EXPLAIN - Query Analysis

In [ ]:
# Using EXPLAIN to analyze queries

explain_example = """
-- EXPLAIN shows how MySQL executes a query
EXPLAIN SELECT * FROM employees WHERE email = 'john@email.com';

-- Key columns in EXPLAIN output:
-- type: ALL (full scan) < index < range < ref < eq_ref < const
-- possible_keys: Which indexes could be used
-- key: Which index is actually used
-- rows: Estimated rows to examine
-- Extra: Additional info (Using index, Using filesort, etc.)
"""

print("EXPLAIN helps you understand query performance!")
print(explain_example)

# You can run EXPLAIN in your MySQL client:
# result = execute_sql("EXPLAIN SELECT * FROM employees WHERE department = 'Engineering'", fetch=True)
# for row in result:
#     print(row)

# 3. Views

A **View** is a virtual table based on a SELECT query. It doesn't store data itself but provides a predefined query.

## Benefits
- Simplify complex queries
- Security (expose only certain columns)
- Abstraction (hide table structure changes)

In [ ]:
# Creating and using Views

view_examples = """
-- Create a simple view
CREATE VIEW employee_directory AS
SELECT emp_id, name, email, department
FROM employees
WHERE is_active = TRUE;

-- Create a view with joins
CREATE VIEW employee_details AS
SELECT 
    e.emp_id,
    e.name,
    d.dept_name,
    m.name AS manager_name
FROM employees e
LEFT JOIN departments d ON e.dept_id = d.dept_id
LEFT JOIN employees m ON e.manager_id = m.emp_id;

-- Use the view like a table
SELECT * FROM employee_directory;
SELECT * FROM employee_details WHERE dept_name = 'Engineering';

-- Update a view (if it's simple enough)
CREATE OR REPLACE VIEW employee_directory AS
SELECT emp_id, name, email, department, hire_date
FROM employees
WHERE is_active = TRUE;

-- Drop a view
DROP VIEW IF EXISTS employee_directory;
"""

print("View SQL Examples:")
print(view_examples)

In [ ]:
# Python example with views

def create_salary_report_view():
    """Create a view for salary reporting."""
    query = """
    CREATE OR REPLACE VIEW salary_report AS
    SELECT 
        d.dept_name,
        COUNT(e.emp_id) AS employee_count,
        AVG(e.salary) AS avg_salary,
        MIN(e.salary) AS min_salary,
        MAX(e.salary) AS max_salary
    FROM departments d
    LEFT JOIN employees e ON d.dept_id = e.dept_id
    GROUP BY d.dept_id, d.dept_name
    """
    execute_sql(query)
    print("View 'salary_report' created!")

def get_salary_report():
    """Query the salary report view."""
    return execute_sql("SELECT * FROM salary_report ORDER BY avg_salary DESC", fetch=True)

print("View functions defined!")

# 4. Stored Procedures

**Stored Procedures** are precompiled SQL code stored in the database. They:
- Accept parameters
- Can contain logic (IF, LOOP, etc.)
- Reduce network traffic
- Centralize business logic

In [ ]:
# Stored Procedure Examples

procedure_examples = """
-- Simple procedure without parameters
DELIMITER //
CREATE PROCEDURE GetAllEmployees()
BEGIN
    SELECT * FROM employees;
END //
DELIMITER ;

-- Call the procedure
CALL GetAllEmployees();


-- Procedure with IN parameter
DELIMITER //
CREATE PROCEDURE GetEmployeesByDept(IN dept_name VARCHAR(50))
BEGIN
    SELECT * FROM employees e
    JOIN departments d ON e.dept_id = d.dept_id
    WHERE d.dept_name = dept_name;
END //
DELIMITER ;

CALL GetEmployeesByDept('Engineering');


-- Procedure with OUT parameter
DELIMITER //
CREATE PROCEDURE GetEmployeeCount(OUT total INT)
BEGIN
    SELECT COUNT(*) INTO total FROM employees;
END //
DELIMITER ;

CALL GetEmployeeCount(@count);
SELECT @count;


-- Procedure with logic
DELIMITER //
CREATE PROCEDURE GiveRaise(
    IN emp_id INT,
    IN raise_percent DECIMAL(5,2),
    OUT new_salary DECIMAL(10,2)
)
BEGIN
    UPDATE employees 
    SET salary = salary * (1 + raise_percent/100)
    WHERE emp_id = emp_id;
    
    SELECT salary INTO new_salary 
    FROM employees 
    WHERE emp_id = emp_id;
END //
DELIMITER ;


-- Drop a procedure
DROP PROCEDURE IF EXISTS GetAllEmployees;
"""

print("Stored Procedure Examples:")
print(procedure_examples)

In [ ]:
# Calling stored procedures from Python

def call_procedure_example():
    """Example of calling stored procedures from Python."""
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor(dictionary=True)
    
    # Call procedure with IN parameter
    cursor.callproc('GetEmployeesByDept', ['Engineering'])
    
    # Get results (stored procs can return multiple result sets)
    for result in cursor.stored_results():
        rows = result.fetchall()
        for row in rows:
            print(row)
    
    # Call procedure with OUT parameter
    args = [0]  # OUT parameter placeholder
    result = cursor.callproc('GetEmployeeCount', args)
    print(f"Total employees: {result[0]}")
    
    cursor.close()
    conn.close()

# Uncomment to run if procedures exist:
# call_procedure_example()

print("Procedure calling function defined!")

# 5. Transactions and ACID

**Transactions** ensure multiple operations complete as a single unit.

## ACID Properties

| Property | Description |
|----------|-------------|
| **Atomicity** | All or nothing - all operations succeed or all fail |
| **Consistency** | Database remains in valid state |
| **Isolation** | Transactions don't interfere with each other |
| **Durability** | Committed changes are permanent |

In [ ]:
# Transaction Examples

transaction_sql = """
-- Start a transaction
START TRANSACTION;

-- Perform operations
UPDATE accounts SET balance = balance - 100 WHERE account_id = 1;
UPDATE accounts SET balance = balance + 100 WHERE account_id = 2;

-- If everything is OK, commit
COMMIT;

-- If something went wrong, rollback
-- ROLLBACK;


-- Savepoints for partial rollback
START TRANSACTION;

INSERT INTO orders (user_id, total) VALUES (1, 100);
SAVEPOINT order_created;

INSERT INTO order_items (order_id, product_id) VALUES (1, 101);
SAVEPOINT items_added;

-- Something went wrong with shipping
ROLLBACK TO SAVEPOINT items_added;

-- Or rollback everything
-- ROLLBACK;

COMMIT;
"""

print("Transaction SQL Examples:")
print(transaction_sql)

In [ ]:
# Python Transaction Examples

def transfer_money(from_account, to_account, amount):
    """Transfer money between accounts with transaction safety."""
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    try:
        # Start transaction (autocommit is False by default)
        conn.start_transaction()
        
        # Check sufficient balance
        cursor.execute(
            "SELECT balance FROM accounts WHERE account_id = %s FOR UPDATE",
            (from_account,)
        )
        balance = cursor.fetchone()[0]
        
        if balance < amount:
            raise ValueError("Insufficient funds")
        
        # Deduct from source
        cursor.execute(
            "UPDATE accounts SET balance = balance - %s WHERE account_id = %s",
            (amount, from_account)
        )
        
        # Add to destination
        cursor.execute(
            "UPDATE accounts SET balance = balance + %s WHERE account_id = %s",
            (amount, to_account)
        )
        
        # All good - commit
        conn.commit()
        print(f"Transferred ${amount} from {from_account} to {to_account}")
        return True
        
    except Exception as e:
        # Something went wrong - rollback
        conn.rollback()
        print(f"Transfer failed: {e}")
        return False
        
    finally:
        cursor.close()
        conn.close()

print("Transaction function defined!")

## 5.1 Isolation Levels

In [ ]:
# Isolation Levels

print("""
ISOLATION LEVELS (from lowest to highest):
==========================================

1. READ UNCOMMITTED
   - Can see uncommitted changes from other transactions
   - "Dirty reads" possible
   - Fastest but least safe

2. READ COMMITTED
   - Only see committed changes
   - No dirty reads
   - "Non-repeatable reads" possible

3. REPEATABLE READ (MySQL default)
   - Consistent reads within transaction
   - "Phantom reads" possible

4. SERIALIZABLE
   - Full isolation
   - Transactions execute one at a time
   - Slowest but safest

-- Set isolation level
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;

-- Check current level
SELECT @@transaction_isolation;
""")

# 6. Triggers

**Triggers** automatically execute SQL code when specific events occur (INSERT, UPDATE, DELETE).

In [ ]:
# Trigger Examples

trigger_examples = """
-- Trigger BEFORE INSERT
DELIMITER //
CREATE TRIGGER before_employee_insert
BEFORE INSERT ON employees
FOR EACH ROW
BEGIN
    SET NEW.created_at = NOW();
    SET NEW.email = LOWER(NEW.email);
END //
DELIMITER ;


-- Trigger AFTER UPDATE - Audit log
DELIMITER //
CREATE TRIGGER after_salary_update
AFTER UPDATE ON employees
FOR EACH ROW
BEGIN
    IF OLD.salary != NEW.salary THEN
        INSERT INTO salary_audit (emp_id, old_salary, new_salary, changed_at)
        VALUES (NEW.emp_id, OLD.salary, NEW.salary, NOW());
    END IF;
END //
DELIMITER ;


-- Trigger to prevent deletion
DELIMITER //
CREATE TRIGGER prevent_admin_delete
BEFORE DELETE ON employees
FOR EACH ROW
BEGIN
    IF OLD.role = 'admin' THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'Cannot delete admin users';
    END IF;
END //
DELIMITER ;


-- View triggers
SHOW TRIGGERS;

-- Drop trigger
DROP TRIGGER IF EXISTS before_employee_insert;
"""

print("Trigger SQL Examples:")
print(trigger_examples)

# 7. Common Table Expressions (CTEs)

**CTEs** create temporary result sets that can be referenced within a query. They improve readability for complex queries.

In [ ]:
# CTE Examples

cte_examples = """
-- Basic CTE
WITH high_earners AS (
    SELECT * FROM employees WHERE salary > 70000
)
SELECT * FROM high_earners ORDER BY salary DESC;


-- Multiple CTEs
WITH 
dept_stats AS (
    SELECT dept_id, AVG(salary) as avg_salary
    FROM employees
    GROUP BY dept_id
),
above_average AS (
    SELECT e.*, ds.avg_salary
    FROM employees e
    JOIN dept_stats ds ON e.dept_id = ds.dept_id
    WHERE e.salary > ds.avg_salary
)
SELECT name, salary, avg_salary FROM above_average;


-- Recursive CTE (for hierarchies)
WITH RECURSIVE org_chart AS (
    -- Base case: top-level managers
    SELECT emp_id, name, manager_id, 1 as level
    FROM employees
    WHERE manager_id IS NULL
    
    UNION ALL
    
    -- Recursive case: employees with managers
    SELECT e.emp_id, e.name, e.manager_id, oc.level + 1
    FROM employees e
    JOIN org_chart oc ON e.manager_id = oc.emp_id
)
SELECT 
    REPEAT('  ', level - 1) || name as org_hierarchy,
    level
FROM org_chart
ORDER BY level, name;
"""

print("CTE Examples:")
print(cte_examples)

# 8. Window Functions

**Window Functions** perform calculations across a set of rows related to the current row.

In [ ]:
# Window Function Examples

window_examples = """
-- ROW_NUMBER: Assign unique row numbers
SELECT 
    name,
    department,
    salary,
    ROW_NUMBER() OVER (ORDER BY salary DESC) as salary_rank
FROM employees;


-- RANK: Rank with gaps for ties
SELECT 
    name,
    department,
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank
FROM employees;


-- Running total
SELECT 
    name,
    hire_date,
    salary,
    SUM(salary) OVER (ORDER BY hire_date) as running_total
FROM employees;


-- Compare to average within partition
SELECT 
    name,
    department,
    salary,
    AVG(salary) OVER (PARTITION BY department) as dept_avg,
    salary - AVG(salary) OVER (PARTITION BY department) as diff_from_avg
FROM employees;


-- LAG/LEAD: Access previous/next row
SELECT 
    name,
    salary,
    LAG(salary) OVER (ORDER BY hire_date) as prev_salary,
    LEAD(salary) OVER (ORDER BY hire_date) as next_salary
FROM employees;
"""

print("Window Function Examples:")
print(window_examples)

# 9. Performance Tips

In [ ]:
print("""
MySQL PERFORMANCE BEST PRACTICES
=================================

1. INDEXING
   - Index columns used in WHERE, JOIN, ORDER BY
   - Don't over-index (slows writes)
   - Use composite indexes for multi-column filters

2. QUERY OPTIMIZATION
   - Use EXPLAIN to analyze queries
   - Avoid SELECT * - specify needed columns
   - Use LIMIT for large result sets
   - Avoid functions on indexed columns in WHERE

3. SCHEMA DESIGN
   - Normalize to 3NF (usually)
   - Use appropriate data types (don't use VARCHAR(255) for everything)
   - Consider denormalization for read-heavy workloads

4. CONNECTION MANAGEMENT
   - Use connection pooling
   - Close connections when done
   - Use persistent connections for web apps

5. CACHING
   - Enable query cache (if appropriate)
   - Use application-level caching (Redis, Memcached)

6. MAINTENANCE
   - Regular ANALYZE TABLE for statistics
   - OPTIMIZE TABLE for fragmented tables
   - Monitor slow query log
""")

# 10. Summary

| Topic | Key Points |
|-------|------------|
| **Indexes** | Speed up reads, slow down writes, use on frequently queried columns |
| **Views** | Virtual tables, simplify queries, provide security |
| **Stored Procedures** | Reusable SQL code, accept parameters, centralize logic |
| **Transactions** | ACID properties, commit/rollback, isolation levels |
| **Triggers** | Auto-execute on INSERT/UPDATE/DELETE, good for audit logs |
| **CTEs** | Temporary result sets, improve readability, support recursion |
| **Window Functions** | Calculations across related rows, ranking, running totals |